# 02 — Predictive coding on image patches

**Background resource:** Section numbers below refer to the [August perspective](../../paper/background/august-perspective.pdf), preserved alongside the current paper.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/brains-to-models-publication/blob/main/code/notebooks/02_predictive_coding.ipynb)

Companion to §3 of *From Brains to Models*.

**Why this notebook.** Cortex spends most of its energy before the world arrives: it continually generates expectations and reacts mainly to what those expectations miss. Predictive coding turns that observation into a concrete mechanism — higher levels send predictions down, lower levels send back only the error — and it is one of the few brain theories precise enough to run in a few lines of code. Building the smallest working version, on patches with natural-image statistics, shows how oriented feature detectors and the idea of "explaining away" fall out of a single rule, and where the analogy to a token predictor stops. A two-level network in the spirit of Rao & Ballard (1999): descending weights predict the layer below; prediction errors drive inference; a Hebbian rule updates the weights.

**Claim.** On the predictive-processing view the brain is also a prediction machine, but it predicts its own sensory input.

**Method.** Two-level Rao–Ballard-style network on $8\times 8$ patches from synthetic $1/f$ images. Seed 7.

**Result.** Oriented first-level bases; residual MSE falls over inference steps.

**Limit.** Not V1, not a person looking at a room, not a foundation model.

<details>
<summary><b>Key terms — for clinicians and neuroscientists</b></summary>

- **[Predictive coding](https://en.wikipedia.org/wiki/Predictive_coding).** The proposal that the brain is a hierarchy of predictions: each level tries to predict the activity of the level below, and only the mismatch is passed upward. Perception becomes the brain's best explanation of its input, not a passive readout.
- **Prediction error.** The part of the input a higher level failed to anticipate — the only signal that needs to travel up. When a level predicts well, its error falls silent and the input is "explained away".
- **Generative model.** An internal model of how hidden causes produce sensory data; the descending predictions are that model running forwards.
- **[Free-energy principle](https://en.wikipedia.org/wiki/Free_energy_principle).** The broader claim that a self-maintaining system perceives and acts so as to minimise long-run surprise; predictive coding is its perceptual special case (named in the paper, not implemented here).
- **[Hebbian](https://en.wikipedia.org/wiki/Hebbian_theory) learning.** A local weight change driven by the correlation of connected units' activity — "cells that fire together wire together" — used here to shape the predictions.

</details>

<details>
<summary><b>Key terms — for the mathematically inclined</b></summary>

- **[Predictive coding](https://en.wikipedia.org/wiki/Predictive_coding).** Perception as gradient descent on a precision-weighted squared-error (Gaussian-surprise) objective: latent states are adjusted until descending predictions match the level below.
- **Rao–Ballard dynamics.** With representation $r$, weights $W$, prediction $\hat{x}=Wr$ and error $e=x-\hat{x}$: states relax by $\dot{r}\propto W^\top e - \lambda\,r$ (inference) and weights update by $\Delta W\propto e\,r^\top$ (a Hebbian/delta rule) on a slower timescale.
- **$1/f$ images.** Synthetic images whose power spectrum falls as $1/f^{\alpha}$, matching the second-order statistics of natural scenes; the network is trained on $8\times 8$ patches.
- **[Free energy](https://en.wikipedia.org/wiki/Free_energy_principle).** Under Gaussian assumptions the summed precision-weighted squared error is a bound on sensory surprisal $-\log p(x)$; minimising it performs inference and learning at once.
- **Read-out.** First-level bases become oriented (edge-like); residual mean-squared error falls across inference steps.

</details>

The same code writes `paper/figures/predictive_coding_patches.pdf` (`make figures`).

### Concepts used in this notebook

This notebook builds a two-level predictive-coding network and trains it on image patches. The two blocks below define the recurring terms once — first in plain language, then in the notation used in the code — so the figures can be read without a separate reference. Both blocks cover the same concepts at a different level of detail.

<details>
<summary><b>Concepts — for clinicians and neuroscientists</b></summary>

- **Hierarchical prediction (top-down) and error (bottom-up).** Two directions of flow: predictions descend, errors ascend. Cortex is largely busy predicting; the senses mainly report what was not predicted.
- **Inference versus learning (two timescales).** *Fast:* with the connections fixed, activity settles to reduce the error for the current image — this is perception. *Slow:* the connections change so predictions improve on average — this is learning. The same error drives both.
- **Receptive field / oriented basis.** After learning, each first-level unit responds best to an edge at a particular orientation — the kind of feature Hubel and Wiesel described in primary visual cortex — here emerging only from predicting patches.
- **Explaining away.** When the model predicts a patch well, the residual error collapses; the input has been accounted for, so little signal travels upward.
- **$1/f$ (natural-image) statistics.** Real scenes carry structure at every spatial scale; the synthetic patches copy that spectrum so the learned features resemble those reported for natural images.
- **Residual (MSE).** The mean squared prediction error — the number that should fall as inference proceeds. It is the model's running "surprise", not a person's.

</details>

<details>
<summary><b>Concepts — for the mathematically inclined</b></summary>

- **State and weights.** Representation $r\in\mathbb{R}^{n}$ and generative weights $W$; prediction $\hat{x}=Wr$, error $e = x-\hat{x}$.
- **Inference.** $\dot{r} = W^\top e - \lambda\,\partial g(r)$: gradient descent on $\tfrac{1}{2}\lVert e\rVert^2$ plus a sparsity/prior term $g$, holding the state responsible for the residual.
- **Learning.** $\Delta W \propto e\,r^\top$: a local, Hebbian-form delta rule applied once inference has partly settled; no global [backpropagation](https://en.wikipedia.org/wiki/Backpropagation) is required.
- **One objective.** Both steps descend the same energy — the precision-weighted squared error — which under Gaussian noise equals variational [free energy](https://en.wikipedia.org/wiki/Free_energy_principle) up to constants.
- **Stimulus.** $8\times 8$ patches drawn from synthetic $1/f$ images; seed 7.
- **What is absent.** No action on the world and no [viability](https://en.wikipedia.org/wiki/Allostasis) constraint; the residual is minimised for a fixed dataset, not to keep an agent alive.

</details>

In [1]:
import os
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO = os.environ.get("BRAINSMODELS_REPO_URL",
                      "https://github.com/arvidl/brains-to-models-publication.git")
COLAB_ROOT = Path("/content/brains-to-models")

if "google.colab" in sys.modules:
    if not (COLAB_ROOT / "code" / "src" / "brainsmodels").is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(COLAB_ROOT)])
    root = COLAB_ROOT
else:
    here = Path.cwd().resolve()
    root = here.parents[1] if here.name == "notebooks" else here

src = root / "code" / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from brainsmodels.repro import DEFAULT_SEED, set_seed
from brainsmodels.predictive_coding import (
    HierarchicalPredictiveCoding,
    PatchDataset,
    render_paper_figure,
    write_figure,
)

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("repo root:", root)
print("seed:", SEED)

repo root: <checkout>
seed: 7


## Patches from 1/f images

Natural scenes have a roughly $1/f$ amplitude spectrum. We synthesize that statistic, mix in a few windowed gratings so a small dictionary has oriented structure to find, cut $8\times 8$ patches, contrast-normalize them, and ZCA-whiten — the usual preprocessing for sparse-coding / predictive-coding demos.

In [2]:
data = PatchDataset.from_pink_noise(seed=SEED)
print(data.train.shape, "whitened training patches")

fig, axes = plt.subplots(2, 8, figsize=(8, 2.4))
for ax, patch in zip(axes.ravel(), data.train_pixel[:16]):
    vmax = np.max(np.abs(patch)) + 1e-8
    ax.imshow(patch.reshape(data.patch_size, data.patch_size), cmap="gray",
              vmin=-vmax, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
fig.suptitle("Training patches (pixel space)", fontsize=11)
fig.tight_layout()
plt.show()

(2400, 64) whitened training patches


/var/folders/1g/_4x26lps6mgg1tr73d7g2bmh0000gp/T/ipykernel_49839/2296392957.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Two-level Rao–Ballard dynamics

Level 1 predicts pixels ($\hat{x} = U_1 r_1$); level 2 predicts $r_1$ ($\hat{r}_1 = U_2 r_2$). Errors $e_0 = x - \hat{x}$ and $e_1 = r_1 - \hat{r}_1$ update the representations, then the weights:

$$
\Delta r_1 \propto U_1^\top e_0 - e_1, \qquad
\Delta r_2 \propto U_2^\top e_1 - \lambda r_2, \qquad
\Delta U \propto e\, r^\top.
$$

In [3]:
model = HierarchicalPredictiveCoding(n_input=data.train.shape[1], seed=SEED)
history = model.train(data.train, n_epochs=25, seed=SEED)

fig, ax = plt.subplots(figsize=(5, 2.6))
ax.plot(np.arange(1, len(history) + 1), history, marker=".")
ax.set_xlabel("Epoch")
ax.set_ylabel("Reconstruction MSE")
ax.set_title("Learning: prediction error on training batches")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

epoch 01/25  recon MSE 0.2998
epoch 02/25  recon MSE 0.2507
epoch 03/25  recon MSE 0.2167
epoch 04/25  recon MSE 0.1951
epoch 05/25  recon MSE 0.1814


epoch 06/25  recon MSE 0.1723


epoch 07/25  recon MSE 0.1663
epoch 08/25  recon MSE 0.1624
epoch 09/25  recon MSE 0.1601


epoch 10/25  recon MSE 0.1588
epoch 11/25  recon MSE 0.1580
epoch 12/25  recon MSE 0.1576
epoch 13/25  recon MSE 0.1573
epoch 14/25  recon MSE 0.1571


epoch 15/25  recon MSE 0.1569


epoch 16/25  recon MSE 0.1568
epoch 17/25  recon MSE 0.1567
epoch 18/25  recon MSE 0.1565


epoch 19/25  recon MSE 0.1565
epoch 20/25  recon MSE 0.1564
epoch 21/25  recon MSE 0.1563
epoch 22/25  recon MSE 0.1563
epoch 23/25  recon MSE 0.1562


epoch 24/25  recon MSE 0.1561


epoch 25/25  recon MSE 0.1560


/var/folders/1g/_4x26lps6mgg1tr73d7g2bmh0000gp/T/ipykernel_49839/1649813066.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Paper figure

Learned first-level bases, then three held-out patches as inference explains them away. Residual MSE is in the whitened space the network actually sees.

In [4]:
fig = render_paper_figure(model, data)
plt.show()

out = write_figure(model, data, root / "paper" / "figures")
print("wrote", out)

/var/folders/1g/_4x26lps6mgg1tr73d7g2bmh0000gp/T/ipykernel_49839/3586766330.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


wrote <checkout>/paper/figures/predictive_coding_patches.pdf
wrote <checkout>/paper/figures/predictive_coding_patches.png
wrote <checkout>/paper/figures/predictive_coding_patches.pdf


## What to notice

- First-level bases become localized and oriented — cheap edge detectors, not pixel templates.
- A few inference steps already reconstruct the patch; later steps mostly clean the residual.
- The objective is the same idea as next-token prediction, but the constraints differ: here the network predicts its own sensory input, not a corpus.

Change `SEED` in the setup cell, or raise `n_level1`, if you want a messier or richer dictionary.

## Why this example is here

This notebook is not a tutorial on vision, and it is not a model of V1. It is a controlled demonstration of the one fact §3 of the paper needs: **the brain, on the predictive-processing view, is also a prediction machine — but what it predicts is its own sensory input, in a body that has to keep going.** Notebook 01 showed next-token prediction on a corpus. Here the same *idea* (minimise surprise) is aimed at a patch of “world,” with a hierarchy and a Hebbian update instead of attention and gradient descent.

The intended readers are the same as the paper’s: clinicians and neuroscientists, and the mathematically inclined who have never trained a network, never whitened an image, and never met “predictive coding.” The two readings below are of the *same* object. Nothing in the code is a patient, and nothing is a theorem about consciousness. The point is to make Rao & Ballard’s loop concrete enough that “the cortex predicts the layer below” is something you can watch, not only cite.

### What the network is asked to do

An $8\times 8$ patch is a vector $x\in\mathbb{R}^{64}$. Two layers of *causes* $r_1,r_2$ and descending weights $U_1,U_2$ try to explain it:

$$
\hat{x}=U_1 r_1,\qquad \hat{r}_1=U_2 r_2.
$$

The mismatches

$$
e_0=x-\hat{x},\qquad e_1=r_1-\hat{r}_1
$$

are **prediction errors**. They do two jobs. First they update the causes (inference, fast):

$$
\Delta r_1 \propto U_1^\top e_0 - e_1,\qquad
\Delta r_2 \propto U_2^\top e_1 - \lambda r_2.
$$

Then they update the weights (learning, slow):

$$
\Delta U \propto e\, r^\top.
$$

That is almost all of it. Higher units predict lower activity; the residual drives both the current explanation and the future dictionary. After a few dozen epochs the first-level columns of $U_1$, plotted back in pixel space, look like small oriented blobs — cheap edge detectors, not photographs of the training images.

There is no label, no class, no “this is a cat.” The only score is how well the network predicts the patch it was just shown.

**For the clinician and the neuroscientist.** This is closer to *explaining away a sensation you already have* than to *classifying a scan*. A radiologist who has seen a thousand chests will “fill in” a rib through a shadow; the surprise is the bit that does not fit. Predictive coding says that fill-in is the descending prediction and the surprise is $e_0$. Extra-classical receptive-field effects (end-stopping, contextual modulation) were Rao & Ballard’s original target: a cell goes quiet when a larger pattern accounts for what a local edge had seemed to be. The toy is that idea at $8\times 8$. It is not a claim that V1 *is* this two-level net, and it is not a claim about MRI, CT, or any clinical image. “Imaging” in this notebook means *natural-image statistics*, not the radiology department.

**For the mathematically inclined.** The pair $(U,r)$ is a linear generative model. Inference is (approximate) minimisation of a squared-error energy

$$
E=\|x-U_1 r_1\|^2+\|r_1-U_2 r_2\|^2+\lambda\|r_2\|^2
$$

by gradient steps in $r$; learning is a Hebbian / outer-product step in $U$ that is the same gradient in the weights. No back-propagation through time is required. Whitening (ZCA) is a linear change of coordinates that makes the input covariance $\approx I$, so the Euclidean residual is a more reasonable proxy for “unexplained structure.” None of this assumes a semantic category. A “basis function” is a column of $U_1$.

### Why $1/f$ images and a few gratings

Natural scenes have a roughly $1/f$ amplitude spectrum: much of the power is in slow, large-scale variation. A purely white-noise patch has nothing for a small dictionary to find. A purely $1/f$ patch is closer to nature but still poor in sharp orientation. Mixing in a few windowed sinusoids gives the dictionary something edge-like to lock onto, without downloading photographs or claiming we have modelled the world.

Four hundred-odd tiny patches, two layers, a laptop. Enough for panel A to show oriented blobs and for panel B to show a residual shrinking. Not a claim that cortex sees pink noise.

**For the clinician and the neuroscientist.** Think of a phantom or a digital test pattern used to check whether a method finds the structure you planted. The gratings are the planted structure. If the bases come out oriented, the loop is doing the job the paper needs; if they came out as pixel dust, it would not be.

**For the mathematically inclined.** The data distribution is a mixture of Gaussian-like $1/f$ fields and localised harmonics, then contrast-normalised and ZCA-whitened. The interesting event is that a bilinear model $U_1 r_1$ develops a Gabor-like dictionary from that mixture *without* a sparsity penalty being the main driver (a light $\lambda$ on $r_2$ is the only explicit regulariser). Olshausen–Field sparse coding is the cousin; Rao–Ballard is the hierarchical, error-driven version.

### How to read the three panels

**Panel A — the dictionary, not a photograph.**  
Each tile is one column of $U_1$, inverted through the whitening so you can see it as a patch. Localised, oriented structure means: the network has found that “an edge here, at this angle” is a reusable cause. A blotchy, global tile would mean it had memorised little photographs. You should see the first kind.

**For the clinician and the neuroscientist.** These are not receptive fields recorded in a cat. They are the *shape the prediction uses*. Hubel & Wiesel’s oriented simple cells are the empirical cousin; this panel is the computational schematic of how such a dictionary could be *learned* by predicting the input.

**For the mathematically inclined.** After ZCA inversion you are looking at $W^{-1}u_k$ (or the analogous map used in the renderer), not at $u_k$ in whitened coordinates. The orientation is a property of those columns, not a label we assigned.

**Panel B — inference is iterative explaining-away.**  
Three held-out patches. Left: the input. Then reconstructions after 1, 5, and 40 inference steps, and the final residual. Early steps already recover the coarse pattern; later steps remove the remaining residual. That is $r$ moving, with $U$ frozen — perception as optimisation, not as a single feed-forward pass.

**For the clinician and the neuroscientist.** The first glance gets the gist; looking longer cleans the error. A mismatch that will not go away is the interesting clinical fact. Here the “mismatch” is just pixels the current $r$ cannot account for.

**For the mathematically inclined.** Each inference step is a discrete gradient (or fixed-point) update of $E$ in $r$. Forty steps is a small inner loop. The residual $e_0$ is the part of $x$ orthogonal, at that moment, to the span implied by the current $r_1$.

**Panel C — the score is prediction-error energy.**  
Mean squared $e_0$ in the whitened space the network actually sees, versus inference step. It falls. That is all “learning to see” means in this toy: the residual got smaller.

**For the clinician and the neuroscientist.** A falling error is not insight, and it is not a diagnosis. It is the same kind of number as a reconstruction loss on an autoencoder. The paper’s claim is about *what is being predicted* (one’s own input) and *why* (a body that must not be surprised into inviability). The curve cannot show the “why.” Notebook 02b is where prediction starts to *act*.

**For the mathematically inclined.** MSE here is $\|e_0\|_2^2$ after whitening, not a proper scoring rule on a discrete alphabet (that was notebook 01). In the infinite-data, well-specified limit one is approximating a linear-Gaussian generative model. Finite patches plus two layers yield a dictionary, not the truth.

### What this is not

It is not a foundation model, not back-propagation, not a convolutional net, and not a simulation of a person looking at a room. Precision weighting, neuromodulation, spikes, and a body are absent. Scaling the same loop — more layers, natural images, a retina — gets you a research programme (predictive processing), not an LLM. The *objective-level* kinship with notebook 01 is real: both minimise a prediction error. The constraints are not: there the error is about the next character of a corpus; here it is about the current sensory vector.

### What this has to do with being human

Section 3’s contrast is not “brains do prediction, therefore they are transformers.” It is that prediction in a cortex is *for* a creature that must keep its sensory states inside a viable bound (the free-energy principle, in Friston’s ambitious form). An LLM predicts tokens from which that bound has been removed. If you can see, in two layers and a pink-noise patch, that “predict the input” is a different *job* from “predict the next token of other people’s text,” then the T-maze in notebook 02b — prediction that *acts* to make a preferred observation come true — has a firm starting point.

Change `SEED` or `n_level1` and the tiles will move. The claim should not: **same idea as next-token prediction, different target, still no body — but now the error is about the world at the sensors, not about a string.**


### Questions for reflection

Not a quiz. Each question can be asked of a brain and of a machine. Write a short answer that a clinician or a neuroscientist, and a mathematically inclined reader, could both test. The two themes are defined in Further reading below.


1. This net and notebook 01 both minimise a prediction error. What is being predicted in each case, and why is that a difference at the computational level, not only in the data type? ([Predictive coding](https://en.wikipedia.org/wiki/Predictive_coding).)

2. Inference updates $r$ with $U$ frozen; learning updates $U$ slowly. In a patient, what is the analogue of “explaining this sensation now” versus “changing what sensations of this kind will mean later”?

3. A falling residual on an $8\times 8$ patch is the score. What would that establish about V1, and what would it not?

4. Whitening makes the Euclidean residual a less silly proxy for unexplained structure. If you skip it, what changes in the mathematics, and what clinical “normalisation” is that roughly like?

5. A one-level net can still shrink $e_0$. What does the second level add that a clinician would call context, and that a mathematician would call a prior on $r_1$?

6. There is no class label. What would adding a diagnostic loss do to the job this teaching model is here to illustrate?

7. The free-energy claim is that prediction is for staying viable ([allostasis](https://en.wikipedia.org/wiki/Allostasis)). Panel C cannot show that “why.” What extra ingredient would have to enter the energy?

8. Oriented blobs resemble simple-cell fields. Agreement of shape is which of Marr’s rows? Why is it not an agreement of hardware?

9. An LLM predicts other people’s tokens; this net predicts its own input. If both are “prediction machines,” what does theme (ii) still isolate? ([Embodiment](https://plato.stanford.edu/entries/embodied-cognition/).)

10. Change `n_level1` and the tiles move. Which claim should not move?


### Further reading

Each annotation is written against the two themes: **(i) From brains to models** asks what a comparison can establish at a computational level: the same objective, the same representation, the same dynamical description, or the same functional test, applied on both sides — brain and model (machine). **(ii) What AI reveals about being human** asks what the comparison isolates when the systems (human and machine) diverge: [embodiment](https://plato.stanford.edu/entries/embodied-cognition/), a body that must satisfy [allostasis](https://en.wikipedia.org/wiki/Allostasis) (viability) constraints, [phenomenal experience](https://plato.stanford.edu/entries/consciousness/), and values that originate in need and [homeostasis](https://en.wikipedia.org/wiki/Homeostasis) rather than in training data.

The demonstrations in this notebook are teaching models. They illustrate a principle at a scale that can be run and modified. They are not recordings from cortex, not evaluations of a large language model, and not evidence that a next-token probability is a value.

Bibliographic links open the paper or a stable copy. Linked terms open a short definition.

1. **Rao, R. P. N., and Ballard, D. H. (1999).** [*Predictive coding in the visual cortex: a functional interpretation of some extra-classical receptive-field effects*](https://homes.cs.washington.edu/~rao/Rao-Ballard-NN-1999.pdf). *Nature Neuroscience* 2, 79–87. Author PDF.
   - *(i) From brains to models.* The recipe of this notebook: feedback carries [predictions](https://en.wikipedia.org/wiki/Predictive_coding) of lower-level activity; feedforward carries the residual. On natural images the hierarchy grew simple-cell-like fields; error units showed end-stopping. Panel A is that dictionary at $8\times 8$; panel B is the residual being explained away.
   - *(ii) What AI reveals about being human.* Rao and Ballard model extra-classical receptive fields, not a person. The cortical claim is hierarchical prediction of sensory input. An LLM predicts tokens in a corpus from which that sensory loop has been removed.

2. **Friston, K. (2010).** [*The free-energy principle: a unified brain theory?*](https://www.fil.ion.ucl.ac.uk/~karl/The%20free-energy%20principle%20A%20unified%20brain%20theory.pdf). *Nature Reviews Neuroscience* 11, 127–138. Author PDF.
   - *(i) From brains to models.* Any self-organising system that persists must minimise a bound on sensory surprisal — variational [free energy](https://en.wikipedia.org/wiki/Free_energy_principle). Perception, action, and learning become one optimisation. The opened review is explicit that unification is a proposal, not an established fact.
   - *(ii) What AI reveals about being human.* The principle is stated for agents that resist disorder. A frozen language model does not minimise free energy in a body. Theme (ii) is that the [viability](https://en.wikipedia.org/wiki/Allostasis) constraint is what the toy cannot add by training longer.

3. **Clark, A. (2013).** [*Whatever next? Predictive brains, situated agents, and the future of cognitive science*](https://doi.org/10.1017/S0140525X12000477). *Behavioral and Brain Sciences* 36(3), 181–204.
   - *(i) From brains to models.* The cognitive-science reading of [predictive processing](https://en.wikipedia.org/wiki/Predictive_coding): a hierarchical generative model, precision-weighted error, perception as controlled inference. This notebook is the visual-coding core of that picture, not the full situated-agent claim.
   - *(ii) What AI reveals about being human.* Clark’s “situated agents” is the remainder. Prediction in a body that acts is not prediction of the next character. [Embodiment](https://plato.stanford.edu/entries/embodied-cognition/) is not an optional wrap on panel C.

4. **Salvatori, T., et al. (2023).** [*A Survey on Brain-inspired Deep Learning via Predictive Coding*](https://arxiv.org/abs/2308.07870). arXiv:2308.07870 (journal version: *Neural Networks* 195, 108161, 2026).
   - *(i) From brains to models.* A recent map of [predictive coding](https://en.wikipedia.org/wiki/Predictive_coding) as a machine-learning method: Rao–Ballard, neural generative coding, biased competition; software and hardware; the claim that PC inverts a generative model by minimising variational free energy, asynchronously, without [backprop](https://en.wikipedia.org/wiki/Backpropagation) as the native rule.
   - *(ii) What AI reveals about being human.* The survey’s hope is more efficient machines. That is theme (i) as engineering. Theme (ii) is unchanged: a PC-trained net on patches is still not a cortex that must keep its prediction error compatible with staying alive.

5. **Schrimpf, M., et al. (2021).** [*The neural architecture of language: Integrative modeling converges on predictive processing*](https://www.pnas.org/doi/10.1073/pnas.2105646118). *PNAS* 118(45), e2105646118.
   - *(i) From brains to models.* On language, next-word transformers — not other objectives — best predict cortical responses. The convergence with this notebook is the *objective* (prediction), not the architecture (attention vs hierarchical error units).
   - *(ii) What AI reveals about being human.* A high brain-score on sentences does not make the Rao–Ballard residual a token. Two prediction machines can share an imperative and still predict different things, for different reasons.

6. **Seth, A. (2021).** [*Being You: A New Science of Consciousness*](https://www.anilseth.com/being-you). Faber & Faber. Author page.
   - *(i) From brains to models.* Places [predictive processing](https://en.wikipedia.org/wiki/Predictive_coding) in a body: interoception and [allostasis](https://en.wikipedia.org/wiki/Allostasis) give biological prediction its urgency, and, on Seth’s reading, its felt quality. The paper uses that claim as the contrast with token prediction, not as a result of this toy.
   - *(ii) What AI reveals about being human.* If prediction of the sensory consequences of action is what experience is *for*, then a disembodied predictor is the wrong kind of object. That is a hypothesis about [consciousness](https://plato.stanford.edu/entries/consciousness/), not a measurement in panel C.


### Exercises

1. **Inference is not learning.** Panel C is the residual falling *inside one patch*, with the weights held fixed. Freeze a trained network, then reconstruct a held-out patch after 1, 5, and 40 inference steps. Then take the *untrained* initial weights and do the same. How much of the falling residual is the hierarchy explaining the patch, and how much is just “any basis plus enough iterations”?

<!-- solution-box: 02_1_inference_not_learning.py -->
<details>
<summary><strong>Show solution</strong> — Inference is not learning</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/02_1_inference_not_learning.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.predictive_coding import HierarchicalPredictiveCoding, PatchDataset
data = PatchDataset.from_pink_noise(n_train=500, n_test=16, n_images=8, seed=SEED)
x = data.test[0]
fresh = HierarchicalPredictiveCoding(n_input=x.size, seed=SEED)
trained = HierarchicalPredictiveCoding(n_input=x.size, seed=SEED)
trained.train(data.train, n_epochs=6, n_infer=20, seed=SEED)

def curve(model, steps=40):
    _, mse = model.infer_history(x, n_steps=steps)
    return np.asarray(mse)

c0, c1 = curve(fresh), curve(trained)
print(f"untrained  MSE t=1 {c0[0]:.3f}  t=40 {c0[-1]:.3f}")
print(f"trained    MSE t=1 {c1[0]:.3f}  t=40 {c1[-1]:.3f}")
print(f"drop untrained {c0[0] - c0[-1]:.3f}   drop trained {c1[0] - c1[-1]:.3f}")

fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.plot(c0, label="untrained weights")
ax.plot(c1, label="after 6 epochs")
ax.set_xlabel("inference step")
ax.set_ylabel("prediction-error MSE")
ax.legend(frameon=False)
ax.set_title("Same infer loop; only the descending weights changed")
plt.tight_layout()
plt.show()
print(
    "Both curves fall: inference is gradient descent on the current error, "
    "and even random columns of U1 span something. The first step is similar "
    "because r starts at zero; the trained curve ends lower. Learning moved "
    "the basis; inference explains the present patch in that basis. This is "
    "still not V1."
)
```

</details>
<!-- /solution-box -->


2. **One level.** The network is two-level: $r_2$ predicts $r_1$, $r_1$ predicts pixels. Train a copy with `n_level2=1` (almost no higher code) and one with the default 12 units, same data and seed. Compare held-out reconstruction MSE after the same number of inference steps. What does the higher level buy on these 8×8 patches, and what can it not buy?

<!-- solution-box: 02_2_one_level.py -->
<details>
<summary><strong>Show solution</strong> — One level</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/02_2_one_level.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.predictive_coding import HierarchicalPredictiveCoding, PatchDataset
data = PatchDataset.from_pink_noise(n_train=500, n_test=24, n_images=8, seed=SEED)
shallow = HierarchicalPredictiveCoding(n_input=data.train.shape[1], n_level2=1, seed=SEED)
deep = HierarchicalPredictiveCoding(n_input=data.train.shape[1], n_level2=12, seed=SEED)
shallow.train(data.train, n_epochs=6, n_infer=20, seed=SEED)
deep.train(data.train, n_epochs=6, n_infer=20, seed=SEED + 1)

def recon_mse(model):
    _, _, tr = model.infer(data.test, n_steps=40, record=True)
    return float(np.mean(tr.error**2))

print(f"n_level2=1   held-out MSE {recon_mse(shallow):.4f}")
print(f"n_level2=12  held-out MSE {recon_mse(deep):.4f}")
print(
    "\nOn 8×8 pink-noise patches the second level is a small extra code for "
    "the first-level coefficients, not a scene. A better reconstruction is "
    "not evidence of a world model, and a flat second level is not evidence "
    "that cortex is one layer. The paper's contrast is the *job*: predict "
    "the current sensory vector, not the next character of a corpus."
)
```

</details>
<!-- /solution-box -->


3. **Skip the whitening.** The descending weights are learned in ZCA-whitened coordinates, then mapped back to pixels for panel A. Train the same architecture on the *unwhitened* pixel patches (`data.train_pixel`). Do the first-level columns still look localized and oriented when you plot them as 8×8 images? What did whitening buy, and what did it not invent?

<!-- solution-box: 02_3_skip_the_whitening.py -->
<details>
<summary><strong>Show solution</strong> — Skip the whitening</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/02_3_skip_the_whitening.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.predictive_coding import HierarchicalPredictiveCoding, PatchDataset
data = PatchDataset.from_pink_noise(n_train=500, n_test=16, n_images=8, seed=SEED)
pix = HierarchicalPredictiveCoding(n_input=data.train_pixel.shape[1], seed=SEED)
zca = HierarchicalPredictiveCoding(n_input=data.train.shape[1], seed=SEED)
pix.train(data.train_pixel, n_epochs=6, n_infer=20, seed=SEED)
zca.train(data.train, n_epochs=6, n_infer=20, seed=SEED)

rf_pix = pix.U1
rf_zca = data.whitening.to_pixel_basis(zca.U1)
energy = rf_zca.reshape(data.patch_size, data.patch_size, -1).var(axis=(0, 1))
order = np.argsort(energy)[::-1][:8]

fig, axes = plt.subplots(2, 8, figsize=(9.5, 2.8))
for col, i in enumerate(order):
    for row, bank, name in ((0, rf_pix, "pixels"), (1, rf_zca, "ZCA→pixels")):
        patch = bank[:, i].reshape(data.patch_size, data.patch_size)
        vmax = np.max(np.abs(patch)) + 1e-8
        axes[row, col].imshow(patch, cmap="gray", vmin=-vmax, vmax=vmax)
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
        if col == 0:
            axes[row, col].set_ylabel(name, fontsize=8)
fig.suptitle("Whitening changes the coordinates the Hebbian rule sees")
plt.tight_layout()
plt.show()
print(
    "Without whitening the columns spend their variance on low-frequency "
    "blobs (the 1/f spectrum). ZCA equalizes those coordinates so a local "
    "rule can spend capacity on oriented structure that is already in the "
    "gratings. Whitening is a change of basis, not a retina, and the "
    "dictionary is still not V1."
)
```

</details>
<!-- /solution-box -->
